In [ ]:
# Uncomment the following to install requirements
#%pip install numpy scipy matplotlib plotly bumps sasdata sasmodels

In [ ]:
from copy import copy, deepcopy
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import bumps.names as bp

from sasdata import data_path
from sasmodels.bumps_model import Experiment, Model
from sasmodels.core import load_model
from sasmodels.data import load_data, plot_data

bp.help()
%matplotlib inline

In [ ]:
bp.help("dream")

In [ ]:
datasets = load_data(str(data_path / '1d_data' / 'latex_smeared.xml'), index='all')
# Plot all the datasets
for data in datasets:
    # Put 2D datasets in their own figure
    if hasattr(data, 'qx_data'):
        plt.figure()
    plot_data(data)
# For latex_smeared.xml use the final dataset (usans data)
data = datasets[-1]
None # end cells with None so that nothing gets printed

In [ ]:
# DEFINE THE MODEL
model_name = "sphere"
pars = dict(
    scale=0.005, background=0.05,
    radius=2200,
    sld=.291, sld_solvent=7.105,
    radius_pd=0.222296,
    )

kernel = load_model(model_name)
model = Model(kernel, **pars)

# SET THE FITTING PARAMETERS
model.radius.range(15, 3000)
model.radius_pd.range(0, 0.5)
model.background.range(0,1)
model.scale.range(0, 1)
# Tie the model to the data
M = Experiment(data=data, model=model, name="usans")
problem = bp.FitProblem(M)

# sans and usans to the same model, though with independent scale and background
sans_model = copy(model)
sans_model.background = bp.Parameter(model.background.value, name="sans background")
sans_model.scale = bp.Parameter(name="sans scale", value=model.scale.value)
sans_model.background.range(0,1)
sans_model.scale.range(0, 1)
M2 = Experiment(data=datasets[0], model=sans_model, name="sans")
sans_problem = problem = bp.FitProblem([M, M2])

#fig = M._plot(backend='plotly')
#fig.show()
# Shouldn't need the matplotlib inline repeated here
#%matplotlib inline
problem.plot()
#print(f"χ² = {problem.chisq_str()}")
print(problem.summarize())

In [ ]:
# Bumps command line options.
!python -m bumps --help

In [ ]:
# Options to fitter should match the fit controls in the bumps command line
opts = dict(method="dream", samples=5000, burn=100, alpha=0.01, trim=True) # quick dream run
#opts = dict(method="dream", samples=20000, burn=500, alpha=0.001, trim=True) # long dream run
#opts = dict(method="amoeba", steps=10)
#opts = dict(method="lm")
result = bp.fit(
    sans_problem,
    session='session.h5',  # store results to a session file
    parallel=0,
    verbose=True,
    **opts,
)
# Note: has to be the first figure in the cell otherwise the sasmodels plotter gets confused
sans_problem.plot()
plt.figure()
bp.plot_convergence(result)
sans_result = result


In [ ]:
# Resume a dream fit to get more statistics. Must be run after the previous cell
opts.update(burn=300, alpha=0.001)
result = bp.fit(
    problem,
    parallel=0,
    verbose=True,
    resume=result,
    **opts,
)
bp.plot_convergence(result)

In [ ]:
#if results.state: results.state.show() # Dream plots and parameter table only
#bp.show_results() # All outputs and plots, including dream plots

In [ ]:
# A sans model will bifurcate if you hold all parameters steady except for sld
# I(q) is scaled by Δρ², so solvent-Δρ and solvent+Δρ are both solutions.
# Modify the fitting parameters and fit ranges so only sld is fit.

# Make sure wwe have the right order of magnitude for scale
print(f"Check that we have the fitted results in the model: χ² = {sans_problem.chisq_str()}")
print(f"sld={float(sans_model.sld):.3f} solvent={float(sans_model.sld_solvent):.3f}")

limited = deepcopy(sans_problem)
# set all parameters to fixed
for p in limited.parameters:
    p.fixed = True
# Because of freevars we aren't allowed to grab individual models using indexing
# Note that sans and usans sld were tied when the model was created.
with limited.push_model(0) as m:
    m.model.sld.range(-1, 20)
limited.model_reset()  # We've updated the fitted parameter set so reset the model
print(limited.summarize())


In [ ]:
# Try to select for and keep multiple solutions
opts = dict(method="dream", init='lhs', samples=20000, burn=200, alpha=0.01, trim=False, outliers='none') # quick dream run

limited.randomize()
bifurcated_results = bp.fit(limited,parallel=0,verbose=True,**opts)
bifurcated_results.state.show()

In [ ]:
def show_bifurcation(problem, result, var, split, corr=False):
    """
    For a problem with bifurcation in one of its parameter histograms,
    plot the statistics around each of the solutions, the one with parameter
    values below the split and the one with parameter values above.
    """
    from numpy import inf
    from bumps.dream.varplot import plot_vars
    from bumps.dream.views import plot_corrmatrix
    from bumps.dream.stats import var_stats, format_vars

    ldraw = result.state.draw(selection={var: (-inf, split)}, portion=1)
    rdraw = result.state.draw(selection={var: (split, inf)}, portion=1)
    lstat, rstat = var_stats(ldraw), var_stats(rdraw)
    ltext = f"Parameter {var} <= {split} for {problem.name or 'problem'}"
    rtext = f"Parameter {var} >= {split} for {problem.name or 'problem'}"
    print(f"=== {ltext} ===")
    print(format_vars(lstat))
    print(f"=== {rtext} ===")
    print(format_vars(rstat))
    fig = plt.figure()
    plot_vars(ldraw, lstat)
    fig.suptitle(ltext)
    if corr:
        plt.figure()
        plot_corrmatrix(ldraw, vstats=lstat)
        fig.suptitle(ltext)
    fig = plt.figure()
    plot_vars(rdraw, rstat)
    fig.suptitle(rtext)
    if corr:
        plt.figure()
        plot_corrmatrix(rdraw, vstats=rstat)
        fig.suptitle(ltext)

# Note: only one parameter histogram because it is only fitting one parameter
show_bifurcation(limited, bifurcated_results, var='sld', split=5)

In [ ]:
# Numbered parameters. Use these when making derived parameter expressions
# Selection, vars and exclude can use either names or numbers
sans_result.state.show_labels()

In [ ]:
# Show an example of derived, hidden and shown parameters
from numpy import pi
from bumps.dream.views import plot_all

draw = result.state.draw(
    portion=1,  # portion to keep, from the right; use 1.0 for all
    derived=lambda p: {'volume': 4/3*pi*p[1]**3},  # derived parameters
    #selection={'radius': (0, 22003),...},  # ranges on individual parameters, or logp
    vars=['radius', 'radius_pd', 'volume', 'background'],  # variables to use for stats and plots
    exclude=['*background*', '*scale*'],  # list nuisance variables you don't want plotted
    thin=1,   # to reduce autocorrelation spikes choose a large thinning value
)

plot_all(draw)

In [ ]:
# Show histogram/correlation plots over the full range

import matplotlib.pyplot as plt
from bumps.dream.views import plot_corrmatrix
from bumps.dream.varplot import var_plot_size, plot_vars
from bumps.dream.stats import var_stats, format_vars

vstats = var_stats(draw)  # parameter statistics
print(format_vars(vstats))  # formatted parameter table

# First plot with 95% interval windowing
plt.figure()
plot_corrmatrix(draw, vstats=vstats, nbins=50)  # correlation plot 95% interval
plt.figure(figsize=var_plot_size(len(vstats)))
plot_vars(draw, vstats, nbins=50)  # parameter histogram 95% interval

# Next plot with the full sample range
plt.figure()
plot_corrmatrix(draw, vstats=vstats, nbins=50, full=True)  # correlation plot with outliers
plt.figure(figsize=var_plot_size(len(vstats)))
plot_vars(draw, vstats, nbins=50, full=True) # parameter histogram with outliers


In [ ]:
# Entropy calculation (requires `pip install scikit-learn`)
from uncertainties import ufloat as U
from bumps.dream.entropy import gmm_entropy

S, Serr = gmm_entropy(draw.points, n_est=10000) # Entropy from draw
print(f"entropy={U(S, Serr):fS} (Gaussian mixture model)")


In [ ]:
# Fit resume for more burn and more samples
def do():
    global results
    opts['burn'] = 500 # if dream, burn an additional 500
    opts['samples'] = max(opts['samples'], 50000)
    results = bp.fit(
        problem,
        #store='session.h5',  # store results to a session file
        parallel=0,
        resume=results,
        verbose=True,
        **opts,
    )
    # Note: has to be the first figure in the cell otherwise the sasmodels plotter gets confused
    problem.plot(p=results.x)
    if results.state:
        results.state.show()

# do() # uncomment this line and rerun the cell to resume the fit. The results variable will be updated

In [ ]:
# First export the fit
bp.export_fit("/tmp/T1", problem, result, basename="test")
# Examine the outputs
!ls /tmp/T1
# Then reload it
# Reload fails because there is no test.py file
# p2, r2 = bp.load_fit_from_export("/tmp/T1/test.par")
# bp.show_table(p2, r2)


In [ ]:
# Save results to a session file
session = Path('/tmp/test.h5').expanduser()
bp.save_fit(session, problem, result, label="example fit")
# Show reloading of state from --session
p2, r2 = bp.load_fit_from_session(session)
bp.show_table(p2, r2)
#do() # Suppress for now... curve example is not installed with bumps on colab

Issues:
* >file >quit on the broswer kills the jupyter kernel

jupyter + webview use cases:
* X display simple fit result in webview
* X load saved session into webview
* X load exported fit into webview
* X wait for current fit to complete
* X grab the results from a completed fit
* X save the fit results to a sesson file
* X export the fit results
* X update parameter vector in webview
* X change fitted parameters and ranges
* repeatedly run a fit, with a check each time to see if it is converged
* run a series of fits, using the results from one to initialize the next
* save problem and result to a cell
* resume a fit
* overlay parameter histograms for same parameter from different experiments
* plot profiles sampled from the posterior

jupyter + session file use cases:
* process a series of session files, producing summary plots on the fitted parameters
* list history entries in a session file
* grab a particular history entry as a results object
* X resume a fit
* combined: loop over history

In [ ]:
# Start the bumps server
await bp.start_bumps()
#bp.display_bumps(height=800) # Fails on colab

In [ ]:
# Send a problem to the running fit server and wait for it to complete
await bp.set_problem(problem, fit=result)
await bp.start_fit_thread(options=dict(fit="dream", samples=30000, burn=200))
problem, result = await bp.get_fit_from_webview(wait=True)
bp.show_table(problem, result)

In [ ]:
# Duplicate a fit, but with synthetic data
problem.name = "original"
problem2 = deepcopy(problem)
problem2.name = "resynth"
problem2.simulate_data() # resynthesize data from the uncertainty
result2 = bp.fit(problem2, fit="dream", samples=30000, burn=200)
bp.show_table(problem2, result2)
#bp.show_results(problem2, result2)

In [ ]:
# Overplot parameter histograms from original and synthtic data
par, nsigma = 1, 4
bins = np.linspace(result.x[par] - nsigma*result.dx[par], result.x[par] + nsigma*result.dx[par], 50)
draw1, draw2 = result.state.draw(), result2.state.draw()
plt.hist(draw1.points[:, par], bins=bins, alpha=0.5, label=f"{problem.name or 'problem 1'}", density=True)
plt.hist(draw2.points[:, par], bins=bins, alpha=0.5, label=f"{problem2.name or 'problem 2'}", density=True)
plt.ylabel(f"P({draw1.labels[par]})")
plt.xlabel(f"{draw1.labels[par]}")
plt.title("Parameter distribution comparison")
plt.legend()
None